In [ ]:
# Potential Vorticity Inversion

In [2]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import get_cmap
import metpy.calc as mpcalc
from metpy.units import units
from numpy import *
import xarray as xr
from netCDF4 import Dataset, num2date
import math
import pygrib
import cdsapi
from datetime import datetime, timedelta
from geopy.distance import geodesic
import numpy as np
import sys
import xarray as xr

In [3]:
# Constant Level
level = 500

In [4]:
### DATA IMPORT ###

# Vorticity Data
filerv_1_31 = '/gdex/data/d633000/e5.oper.an.pl/202601/e5.oper.an.pl.128_138_vo.ll025sc.2026013100_2026013123.nc' # relative vorticity
filerv_2_1 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_138_vo.ll025sc.2026020100_2026020123.nc' 
filerv_2_2 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_138_vo.ll025sc.2026020200_2026020223.nc'

drv_1_31 = xr.open_dataset(filerv_1_31) #.metpy.parse_cf()
drv_2_1 = xr.open_dataset(filerv_2_1)
drv_2_2 = xr.open_dataset(filerv_2_2)

filepv_1_31 = '/gdex/data/d633000/e5.oper.an.pl/202601/e5.oper.an.pl.128_060_pv.ll025sc.2026013100_2026013123.nc' # potential vorticity
filepv_2_1 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_060_pv.ll025sc.2026020100_2026020123.nc'
filepv_2_2 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_060_pv.ll025sc.2026020200_2026020223.nc'

dpv_1_31 = xr.open_dataset(filepv_1_31)
dpv_2_1 = xr.open_dataset(filepv_2_1)
dpv_2_2 = xr.open_dataset(filepv_2_2)

# Wind Data
fileuwnd_1_31 = '/gdex/data/d633000/e5.oper.an.pl/202601/e5.oper.an.pl.128_131_u.ll025uv.2026013100_2026013123.nc' # u component
fileuwnd_2_1 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_131_u.ll025uv.2026020100_2026020123.nc'
fileuwnd_2_2 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_131_u.ll025uv.2026020200_2026020223.nc'

du_1_31 = xr.open_dataset(fileuwnd_1_31)
du_2_1 = xr.open_dataset(fileuwnd_2_1)
du_2_2 = xr.open_dataset(fileuwnd_2_2)

filevwnd_1_31 = '/gdex/data/d633000/e5.oper.an.pl/202601/e5.oper.an.pl.128_132_v.ll025uv.2026013100_2026013123.nc' # v component
filevwnd_2_1 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_132_v.ll025uv.2026020100_2026020123.nc'
filevwnd_2_2 = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_132_v.ll025uv.2026020200_2026020223.nc'

dv_1_31 = xr.open_dataset(filevwnd_1_31)
dv_2_1 = xr.open_dataset(filevwnd_2_1)
dv_2_2 = xr.open_dataset(filevwnd_2_2)

# pre looping through whole storm
fileuwnd = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_131_u.ll025uv.2026020100_2026020123.nc' 
filevwnd = '/gdex/data/d633000/e5.oper.an.pl/202602/e5.oper.an.pl.128_132_v.ll025uv.2026020100_2026020123.nc' 

In [5]:
### CONCATENATE ###

drv_1_31 = drv_1_31.VO.metpy.sel(level=level)
drv_2_1 = drv_2_1.VO.metpy.sel(level=level)
drv_2_2 = drv_2_2.VO.metpy.sel(level=level)
print(drv_1_31)

'''
pv_storm = xr.concat([dpv_1_31, dpv_2_1, dpv_2_2], dim='time')
rv_storm = xr.concat([drv_1_31, drv_2_1, drv_2_2], dim='time')
'''


vor = rv_storm.VO # relative vorticity
rv_storm = rv_storm.VO.metpy.sel(level=level)
vor2d = rv_storm.isel(time=0)
print(vor2d)

'''
vor = pv_storm.PV # potential vorticity
pv_storm = pv_storm.PV.metpy.sel(level=level)
vor2d = pv_storm.isel(time=0)
'''

<xarray.DataArray 'VO' (time: 24, latitude: 721, longitude: 1440)> Size: 100MB
[24917760 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 192B 2026-01-31 ... 2026-01-31T23:00:00
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    level      float64 8B 500.0
Attributes: (12/15)
    long_name:           Vorticity (relative)
    short_name:          vo
    units:               s**-1
    original_format:     WMO GRIB 1 with ECMWF local table
    ecmwf_local_table:   128
    ecmwf_parameter:     138
    ...                  ...
    rda_dataset:         ds633.0
    rda_dataset_url:     https:/rda.ucar.edu/datasets/ds633.0/
    rda_dataset_doi:     DOI: 10.5065/BH6N-5N20
    rda_dataset_group:   ERA5 atmospheric pressure level analysis [netCDF4]
    quantization:        quantization_info
    quantization_nsd:    7


NameError: name 'rv_storm' is not defined

In [ ]:
### POISSON ###

from xinvert import invert_Poisson

iParams = {
    'BCs'      : ['extend', 'periodic'],
    'mxLoop'   : 1000,
    'tolerance': 1e-12,
}

sf = invert_Poisson(vor2d, dims=['latitude','longitude'], iParams=iParams)